# CAR-T eligibility-aware scheduling — compact reformulation

A compact, assignment-style formulation of the **same problem** as the flow model in
`CART_eligibility_US.ipynb` (equations (1)–(22) of *Updated Model_Patient tiers_revised_v3_pure.docx*).
This notebook is self-contained: it shares no code with the flow-model notebook and only **reads** the
existing `results_us*` folders for the final comparison.

**Variable.** `X[p, m, j1, j2, t] = 1` if patient *p* goes to facility *m*, outbound mode *j1*, return mode *j2*,
and manufacturing starts on day *t*.

**Precomputed for every combination**

| quantity | formula |
|---|---|
| arrival | `a_p + T_LS + TT1[c(p), m, j1]` |
| wait | `t − arrival` |
| delivery | `t + T_MFE + T_QC + TT2[m, h(p), j2]` |
| cost | `CVM[m] + TT1·U1[c,m,j1] + TT2·U2[m,h,j2] + HC·wait` |
| riskD, riskU | `pi[g(p), delivery − a_p, D]`, `pi[g(p), delivery − a_p, U]` |

`X` exists only when `t ≥ arrival`, `delivery − a_p ≤ ND` and `delivery ≤ NT`.

**Constraints**
- (R1) every patient gets exactly one plan.
- (R2) for every facility *m* and day *t*: plans starting in `[t − T_MFE + 1, t]` ≤ `FCAP[m]`.

**Objectives.** Baseline: `min Σ X·cost`. Patient-aware: `min Σ X·cost + λ_D · Σ X·riskD` (λ_D = 150,000, λ_U = 0).

In [1]:
import os, re, time, itertools
import numpy as np
import pandas as pd
import openpyxl
import pyomo.environ as pyo

OUT_DIR = "results_reformulation"
os.makedirs(os.path.join(OUT_DIR, "plans"), exist_ok=True)
pd.set_option("display.width", 260, "display.max_columns", 40, "display.max_rows", 200)

LAMBDA_D, LAMBDA_U = 150_000.0, 0.0
CLASS_COLUMN = "g(p) at 30%"          # Patients column I

# scenario label -> (workbook, folder with the flow-model results; read-only, used only for the comparison)
SCENARIOS = {
    "Houston 10 / T_MFE 7": ("CART_instance_30patients.xlsx",             "results_us"),
    "Houston 4 / T_MFE 7":  ("CART_instance_30patients_cap4.xlsx",        "results_us_cap4"),
    "Houston 2 / T_MFE 7":  ("CART_instance_30patients_cap2.xlsx",        "results_us_cap2"),
    "Houston 4 / T_MFE 14": ("CART_instance_30patients_cap4_tmfe14.xlsx", "results_us_cap4_tmfe14"),
    "Houston 2 / T_MFE 14": ("CART_instance_30patients_cap2_tmfe14.xlsx", "results_us_cap2_tmfe14"),
    "Houston 10 / T_MFE 5": ("CART_instance_30patients_cap10_tmfe5.xlsx", "results_us_cap10_tmfe5"),
}
# Size of the flow model (free variables, constraints) for each scenario. These were printed by the
# CART_eligibility_US.ipynb runs ("Model built: ... free variables, ... constraints") but are not stored in the
# results_us* folders, so they are recorded here from those printed outputs (notebook history on Model_Code_version_3).
FLOW_MODEL_SIZE = {
    "Houston 10 / T_MFE 7": (598_203, 440_409), "Houston 4 / T_MFE 7":  (598_203, 440_409),
    "Houston 2 / T_MFE 7":  (598_203, 440_409), "Houston 4 / T_MFE 14": (597_573, 440_409),
    "Houston 2 / T_MFE 14": (597_573, 440_409), "Houston 10 / T_MFE 5": (598_383, 440_409),
}
MODELS = ["Baseline", "Patient-aware"]
slug = lambda s: re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")

## 1. Data loader (workbook → plain Python data, nothing hard-coded)

In [2]:
def load_instance(path):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=False)

    def table(sheet, first_row, ncols, prefix):
        out = []
        for r in wb[sheet].iter_rows(min_row=first_row, max_col=ncols, values_only=True):
            if not (isinstance(r[0], str) and re.fullmatch(prefix + r"\d+", r[0])):
                break
            out.append(r)
        return out

    settings = {str(lab): val for lab, val, *_ in wb["Settings"].iter_rows(min_row=4, max_col=3, values_only=True) if lab}
    get = lambda start: next(v for k, v in settings.items() if k.startswith(start))
    d = dict(path=path, ND=int(get("ND")), T_LS=int(get("T_LS")), T_MFE=int(get("T_MFE")), T_QC=int(get("T_QC")),
             HC=float(get("HC")), NT=int(get("Horizon NT")))

    fac = table("Facilities", 4, 5, "m")
    d["M"] = [r[0] for r in fac]
    d["M_CITY"] = {r[0]: r[1] for r in fac}
    d["FCAP"] = {r[0]: int(r[3]) for r in fac}
    d["CVM"] = {r[0]: float(r[4]) for r in fac}

    sites = table("Sites", 4, 5, "c")
    d["PAIRED_H"] = {r[0]: r[3] for r in sites}                      # MAP[c,h] = 1 iff h is the paired hospital
    d["SITE_CITY"] = {r[0]: r[1] for r in sites}

    out_rows = table("Transport_out", 4, 7, "c")                      # (c, m, j): Days = col F, U = col G
    ret_rows = table("Transport_return", 4, 7, "m")                   # (m, h, j)
    d["TT1"] = {(r[0], r[1], r[2]): int(r[5]) for r in out_rows}
    d["U1"] = {(r[0], r[1], r[2]): float(r[6]) for r in out_rows}
    d["TT2"] = {(r[0], r[1], r[2]): int(r[5]) for r in ret_rows}
    d["U2"] = {(r[0], r[1], r[2]): float(r[6]) for r in ret_rows}
    d["J"] = sorted({k[2] for k in d["TT1"]} | {k[2] for k in d["TT2"]})
    d["MODE"] = {"j1": "fast", "j2": "slow"}                          # Modes sheet: U fast (j1), U slow (j2)

    header = [c.value for c in wb["Patients"][4]]
    gcol = header.index(CLASS_COLUMN)
    assert gcol == 8, "class column is expected in column I"
    pats = table("Patients", 5, 9, "p")
    d["P"] = [r[0] for r in pats]
    d["A"] = {r[0]: int(r[1]) for r in pats}                          # collection day a_p
    d["SITE"] = {r[0]: r[2] for r in pats}
    d["G"] = {r[0]: r[gcol] for r in pats}                            # class at collection
    d["HOSP"] = {r[0]: d["PAIRED_H"][r[2]] for r in pats}             # h(p)

    q = {str(r[0]).split(":")[0]: float(r[1]) for r in wb["Clinical"].iter_rows(min_row=4, max_row=6, max_col=2, values_only=True)}
    Pm = np.array([[1 - q["q_RU"] - q["q_RD"], q["q_RU"], q["q_RD"]],
                   [0.0, 1 - q["q_UD"], q["q_UD"]],
                   [0.0, 0.0, 1.0]])                                  # eq. (21), order (R, U, D)
    pi, Pk = {}, np.eye(3)
    for tau in range(d["NT"] + 1):
        for gi, g in enumerate("RU"):
            for si, s in enumerate("RUD"):
                pi[g, tau, s] = float(Pk[gi, si])
        Pk = Pk @ Pm
    d["PI"] = pi
    # cross-check against the Pi sheet (tolerance 1e-6)
    dev = 0.0
    for r in wb["Pi"].iter_rows(min_row=5, max_col=7, values_only=True):
        if not isinstance(r[0], (int, float)):
            break
        if int(r[0]) <= d["NT"]:
            for (g, s), v in zip([("R", "R"), ("R", "U"), ("R", "D"), ("U", "R"), ("U", "U"), ("U", "D")], r[1:7]):
                dev = max(dev, abs(pi[g, int(r[0]), s] - v))
    assert dev <= 1e-6, f"pi does not match the Pi sheet ({dev})"
    d["pi_max_dev"] = dev
    return d


inst = {name: load_instance(wbk) for name, (wbk, _) in SCENARIOS.items()}
print(pd.DataFrame({name: dict(workbook=d["path"], ND=d["ND"], NT=d["NT"], T_LS=d["T_LS"], T_MFE=d["T_MFE"], T_QC=d["T_QC"],
                               HC=d["HC"], lines=" / ".join(f"{d['M_CITY'][m].split(',')[0]} {d['FCAP'][m]}" for m in d["M"]),
                               patients=len(d["P"]), urgent=sum(g == "U" for g in d["G"].values()))
                    for name, d in inst.items()}).T.to_string())

                                                       workbook  ND  NT T_LS T_MFE T_QC      HC                                         lines patients urgent
Houston 10 / T_MFE 7              CART_instance_30patients.xlsx  40  52    1     7    7  177.22  Philadelphia 4 / Houston 10 / Los Angeles 10       30      9
Houston 4 / T_MFE 7          CART_instance_30patients_cap4.xlsx  40  52    1     7    7  177.22   Philadelphia 4 / Houston 4 / Los Angeles 10       30      9
Houston 2 / T_MFE 7          CART_instance_30patients_cap2.xlsx  40  52    1     7    7  177.22   Philadelphia 4 / Houston 2 / Los Angeles 10       30      9
Houston 4 / T_MFE 14  CART_instance_30patients_cap4_tmfe14.xlsx  40  52    1    14    7  177.22   Philadelphia 4 / Houston 4 / Los Angeles 10       30      9
Houston 2 / T_MFE 14  CART_instance_30patients_cap2_tmfe14.xlsx  40  52    1    14    7  177.22   Philadelphia 4 / Houston 2 / Los Angeles 10       30      9
Houston 10 / T_MFE 5  CART_instance_30patients_cap10

## 2. Plan enumeration and model

In [3]:
def enumerate_plans(d):
    """All admissible (p, m, j1, j2, t) with their precomputed attributes."""
    plans = {}
    for p in d["P"]:
        c, h, a, g = d["SITE"][p], d["HOSP"][p], d["A"][p], d["G"][p]
        for m, j1, j2 in itertools.product(d["M"], d["J"], d["J"]):
            tt1, tt2 = d["TT1"][c, m, j1], d["TT2"][m, h, j2]
            arrival = a + d["T_LS"] + tt1
            for t in range(arrival, d["NT"] + 1):                                  # t >= arrival
                delivery = t + d["T_MFE"] + d["T_QC"] + tt2
                if delivery - a > d["ND"] or delivery > d["NT"]:                   # TRT <= ND, inside the horizon
                    break
                wait = t - arrival
                plans[p, m, j1, j2, t] = dict(
                    arrival=arrival, wait=wait, delivery=delivery, TRT=delivery - a,
                    cost=d["CVM"][m] + tt1 * d["U1"][c, m, j1] + tt2 * d["U2"][m, h, j2] + d["HC"] * wait,
                    riskD=d["PI"][g, delivery - a, "D"], riskU=d["PI"][g, delivery - a, "U"])
    return plans


def build_model(d, plans):
    mdl = pyo.ConcreteModel()
    mdl.K = pyo.Set(initialize=list(plans), dimen=5, ordered=True)
    mdl.X = pyo.Var(mdl.K, domain=pyo.Binary)
    by_p, by_mt = {}, {}
    for k in plans:
        by_p.setdefault(k[0], []).append(k)
        by_mt.setdefault((k[1], k[4]), []).append(k)
    # (R1) exactly one plan per patient
    mdl.R1 = pyo.Constraint(d["P"], rule=lambda mdl, p: sum(mdl.X[k] for k in by_p[p]) == 1)
    # (R2) capacity: plans started in [t - T_MFE + 1, t] occupy a line on day t
    def r2(mdl, m, t):
        ks = [k for tt in range(max(1, t - d["T_MFE"] + 1), t + 1) for k in by_mt.get((m, tt), [])]
        return sum(mdl.X[k] for k in ks) <= d["FCAP"][m] if ks else pyo.Constraint.Skip
    mdl.R2 = pyo.Constraint(d["M"], range(1, d["NT"] + 1), rule=r2)
    mdl.cost = pyo.Expression(expr=sum(plans[k]["cost"] * mdl.X[k] for k in plans))
    mdl.ED = pyo.Expression(expr=sum(plans[k]["riskD"] * mdl.X[k] for k in plans))
    mdl.EU = pyo.Expression(expr=sum(plans[k]["riskU"] * mdl.X[k] for k in plans))
    mdl.obj = {"Baseline": pyo.Objective(expr=mdl.cost, sense=pyo.minimize),
               "Patient-aware": pyo.Objective(expr=mdl.cost + LAMBDA_D * mdl.ED + LAMBDA_U * mdl.EU, sense=pyo.minimize)}
    for name, o in mdl.obj.items():
        mdl.add_component("obj_" + slug(name), o)
        o.deactivate()
    return mdl


def solve(mdl, which):
    for o in mdl.obj.values():
        o.deactivate()
    mdl.obj[which].activate()
    opt = pyo.SolverFactory("appsi_highs")
    opt.options.update({"mip_rel_gap": 0.0, "mip_abs_gap": 1e-9, "threads": 4, "random_seed": 0, "time_limit": 1800})
    t0 = time.time()
    try:
        res = opt.solve(mdl, tee=False)
    except RuntimeError as e:
        return None, time.time() - t0, str(e).splitlines()[0]
    tc = res.solver.termination_condition
    return (tc == pyo.TerminationCondition.optimal), time.time() - t0, str(tc)

## 3. Solve the six scenarios (Baseline and Patient-aware) and verify every solution

In [4]:
def plan_table(d, plans, mdl):
    rows = []
    for k in plans:
        if pyo.value(mdl.X[k]) > 0.5:
            p, m, j1, j2, t = k
            a = plans[k]
            rows.append({"p": p, "class": d["G"][p], "site": d["SITE"][p], "site city": d["SITE_CITY"][d["SITE"][p]],
                         "hospital": d["HOSP"][p], "facility": m, "out mode": d["MODE"][j1], "return mode": d["MODE"][j2],
                         "collection day": d["A"][p], "arrival": a["arrival"], "start": t, "delivery": a["delivery"],
                         "TRT": a["TRT"], "W_p": a["wait"], "pi_D at delivery": a["riskD"], "pi_U at delivery": a["riskU"],
                         "plan cost": a["cost"], "_j1": j1, "_j2": j2})
    return pd.DataFrame(rows).sort_values("p", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)


def verify(d, tab, mdl):
    """Checks recomputed from the raw data and the chosen plans (not from the model's own coefficients)."""
    out = []
    out.append(("one plan per patient", sorted(tab.p) == sorted(d["P"]) and tab.p.is_unique))
    load = {(m, t): int(((tab.facility == m) & (tab.start >= t - d["T_MFE"] + 1) & (tab.start <= t)).sum())
            for m in d["M"] for t in range(1, d["NT"] + 1)}
    out.append(("capacity respected for every m and t", all(load[m, t] <= d["FCAP"][m] for m, t in load)))
    out.append((f"TRT <= ND ({d['ND']}) for every patient", bool((tab.TRT <= d["ND"]).all())))
    tt1 = tab.apply(lambda r: d["TT1"][r.site, r.facility, r._j1], axis=1)
    tt2 = tab.apply(lambda r: d["TT2"][r.facility, r.hospital, r._j2], axis=1)
    out.append(("timing recomputed from route/modes (arrival, start >= arrival, delivery, TRT, horizon)",
                bool(((tab.arrival == tab["collection day"] + d["T_LS"] + tt1) & (tab.start >= tab.arrival)
                      & (tab.delivery == tab.start + d["T_MFE"] + d["T_QC"] + tt2)
                      & (tab.TRT == d["T_LS"] + tt1 + tab.W_p + d["T_MFE"] + d["T_QC"] + tt2)
                      & (tab.delivery <= d["NT"])).all())))
    ed = sum(d["PI"][r["class"], r.delivery - r["collection day"], "D"] for _, r in tab.iterrows())
    eu = sum(d["PI"][r["class"], r.delivery - r["collection day"], "U"] for _, r in tab.iterrows())
    out.append(("E_D recomputed from the chosen plans = solver value (1e-6)", abs(ed - pyo.value(mdl.ED)) <= 1e-6))
    out.append(("E_U recomputed from the chosen plans = solver value (1e-6)", abs(eu - pyo.value(mdl.EU)) <= 1e-6))
    u1 = tab.apply(lambda r: d["U1"][r.site, r.facility, r._j1], axis=1)
    u2 = tab.apply(lambda r: d["U2"][r.facility, r.hospital, r._j2], axis=1)
    cost = (tab.facility.map(d["CVM"]) + tt1 * u1 + tt2 * u2 + d["HC"] * tab.W_p).sum()
    out.append(("total cost recomputed from the chosen plans = solver value", abs(cost - pyo.value(mdl.cost)) <= 1e-4))
    return out


RES, PLANS, VERIF, SIZE = {}, {}, {}, {}
for name, d in inst.items():
    plans = enumerate_plans(d)
    t0 = time.time()
    mdl = build_model(d, plans)
    SIZE[name] = dict(variables=len(plans), constraints=len(mdl.R1) + len(mdl.R2), build_s=time.time() - t0)
    print(f"\n{name}: {SIZE[name]['variables']:,} binary variables, {SIZE[name]['constraints']} constraints "
          f"(R1 {len(mdl.R1)}, R2 {len(mdl.R2)})")
    for which in MODELS:
        ok, secs, status = solve(mdl, which)
        if not ok:
            RES[name, which] = dict(status="infeasible" if ok is None else status)
            print(f"  {which:<14s} NOT SOLVED: {status}")
            continue
        tab = plan_table(d, plans, mdl)
        PLANS[name, which] = tab
        VERIF[name, which] = verify(d, tab, mdl)
        tab.drop(columns=["_j1", "_j2"]).to_csv(os.path.join(OUT_DIR, "plans", f"{slug(name)}__{which}.csv"), index=False)
        RES[name, which] = dict(status="optimal", objective=pyo.value(mdl.obj[which]), cost=pyo.value(mdl.cost),
                                E_D=pyo.value(mdl.ED), E_U=pyo.value(mdl.EU), solve_s=secs)
        r = RES[name, which]
        print(f"  {which:<14s} objective={r['objective']:>14,.4f}  cost={r['cost']:>12,.2f}  E_D={r['E_D']:.6f}  "
              f"E_U={r['E_U']:.6f}  [{secs:.2f}s]  verification: {'PASS' if all(o for _, o in VERIF[name, which]) else 'FAIL'}")


Houston 10 / T_MFE 7: 7,908 binary variables, 144 constraints (R1 30, R2 114)


  Baseline       objective=  384,646.9000  cost=  384,646.90  E_D=4.658131  E_U=8.836567  [0.39s]  verification: PASS


  Patient-aware  objective=1,035,503.5336  cost=  393,745.96  E_D=4.278384  E_U=9.247112  [0.23s]  verification: PASS



Houston 4 / T_MFE 7: 7,908 binary variables, 144 constraints (R1 30, R2 114)


  Baseline       objective=  386,896.4400  cost=  386,896.44  E_D=4.923437  E_U=9.077753  [0.35s]  verification: PASS


  Patient-aware  objective=1,068,850.9947  cost=  403,728.80  E_D=4.434148  E_U=9.359367  [0.26s]  verification: PASS

Houston 2 / T_MFE 7: 7,908 binary variables, 144 constraints (R1 30, R2 114)


... (5 outputs left out)

  Baseline       objective=  441,561.6200  cost=  441,561.62  E_D=7.022512  E_U=8.657708  [0.33s]  verification: PASS


  Patient-aware  objective=1,468,360.0565  cost=  450,612.94  E_D=6.784981  E_U=9.255986  [0.26s]  verification: PASS

Houston 10 / T_MFE 5: 8,628 binary variables, 144 constraints (R1 30, R2 114)


  Baseline       objective=  377,999.3000  cost=  377,999.30  E_D=4.142648  E_U=9.183097  [0.30s]  verification: PASS


  Patient-aware  objective=  950,014.0857  cost=  388,177.76  E_D=3.745576  E_U=9.251571  [0.19s]  verification: PASS


## 4. Six-scenario table (reformulation)

In [5]:
ROWS = ["E_D, Baseline", "E_D, Patient-aware", "Difference (Baseline − Patient-aware)",
        "Relative reduction (Difference ÷ E_D Baseline)", "Extra cost, Patient-aware vs Baseline (USD)",
        "Mean wait, Baseline (days)", "Max wait, Baseline (days)",
        "Mean wait of urgent patients, Baseline (days)", "Mean wait of urgent patients, Patient-aware (days)"]


def six_rows(ed_b, ed_a, cost_b, cost_a, tab_b, tab_a):
    return dict(zip(ROWS, [ed_b, ed_a, ed_b - ed_a, (ed_b - ed_a) / ed_b, cost_a - cost_b,
                           tab_b["W_p"].mean(), tab_b["W_p"].max(),
                           tab_b.loc[tab_b["class"] == "U", "W_p"].mean(), tab_a.loc[tab_a["class"] == "U", "W_p"].mean()]))


six = {}
for name in SCENARIOS:
    b, a = RES[name, "Baseline"], RES[name, "Patient-aware"]
    if b["status"] != "optimal" or a["status"] != "optimal":
        six[name] = dict.fromkeys(ROWS, "infeasible")
        continue
    six[name] = six_rows(b["E_D"], a["E_D"], b["cost"], a["cost"], PLANS[name, "Baseline"], PLANS[name, "Patient-aware"])
six = pd.DataFrame(six).loc[ROWS]
six.to_csv(os.path.join(OUT_DIR, "six_scenarios_reformulation.csv"), index_label="metric")
print(six.to_string(float_format=lambda x: f"{x:,.4f}"))

                                                    Houston 10 / T_MFE 7  Houston 4 / T_MFE 7  Houston 2 / T_MFE 7  Houston 4 / T_MFE 14  Houston 2 / T_MFE 14  Houston 10 / T_MFE 5
E_D, Baseline                                                     4.6581               4.9234               5.2633                6.7800                7.0225                4.1426
E_D, Patient-aware                                                4.2784               4.4341               4.5644                6.5381                6.7850                3.7456
Difference (Baseline − Patient-aware)                             0.3797               0.4893               0.6989                0.2420                0.2375                0.3971
Relative reduction (Difference ÷ E_D Baseline)                    0.0815               0.0994               0.1328                0.0357                0.0338                0.0958
Extra cost, Patient-aware vs Baseline (USD)                   9,099.0600          16,832.3600  

## 5. Flow model vs reformulation
The flow-model figures are read from the existing `results_us*` folders (saved CSVs; nothing is re-solved).
Flow objective: Baseline = TOTCOST; Patient-aware = TOTCOST + λ_D·E_D.

In [6]:
def read_flow(folder):
    sm = pd.read_csv(os.path.join(folder, "summary_runs.csv")).set_index("run")
    pick = lambda names: sm.loc[next(n for n in names if n in sm.index)]
    path = lambda names: next(os.path.join(folder, n) for n in names if os.path.exists(os.path.join(folder, n)))
    rows = {"Baseline": pick(["Baseline", "baseline"]), "Patient-aware": pick(["Patient-aware", "lambda_D=150,000"])}
    tabs = {"Baseline": pd.read_csv(path(["patients_Baseline.csv", "patients_baseline.csv"])),
            "Patient-aware": pd.read_csv(path(["patients_Patient-aware.csv", "patients_lambdaD_150k.csv"]))}
    out = {}
    for which in MODELS:
        r = rows[which]
        out[which] = dict(cost=float(r.TOTCOST), E_D=float(r.E_D), solve_s=float(r.solve_s), plan=tabs[which],
                          objective=float(r.TOTCOST) + (LAMBDA_D * float(r.E_D) if which == "Patient-aware" else 0.0))
    return out


FLOW = {name: read_flow(folder) for name, (_, folder) in SCENARIOS.items()}

cmp_rows = []
for name in SCENARIOS:
    for which in MODELS:
        f, r = FLOW[name][which], RES[name, which]
        cmp_rows.append({"scenario": name, "model": which,
                         "objective, flow": f["objective"], "objective, reformulation": r.get("objective", np.nan),
                         "total cost, flow": f["cost"], "total cost, reformulation": r.get("cost", np.nan),
                         "E_D, flow": f["E_D"], "E_D, reformulation": r.get("E_D", np.nan),
                         "variables, flow": FLOW_MODEL_SIZE[name][0], "variables, reformulation": SIZE[name]["variables"],
                         "constraints, flow": FLOW_MODEL_SIZE[name][1], "constraints, reformulation": SIZE[name]["constraints"],
                         "solve time (s), flow": f["solve_s"], "solve time (s), reformulation": r.get("solve_s", np.nan)})
cmp = pd.DataFrame(cmp_rows)
cmp.to_csv(os.path.join(OUT_DIR, "comparison_flow_vs_reformulation.csv"), index=False)
show = cmp.copy()
for c in show.columns:
    if c.startswith(("objective", "total cost")):
        show[c] = show[c].map("{:,.2f}".format)
    elif c.startswith("E_D"):
        show[c] = show[c].map("{:.6f}".format)
    elif c.startswith("solve"):
        show[c] = show[c].map("{:.2f}".format)
    elif c.startswith(("variables", "constraints")):
        show[c] = show[c].map("{:,}".format)
print(show.set_index(["scenario", "model"]).T.to_string())
print("\nSolve time = wall time of the solver call (Pyomo translation + HiGHS). Flow times are those recorded in the "
      "earlier runs; model sizes are before presolve.")

scenario                      Houston 10 / T_MFE 7               Houston 4 / T_MFE 7               Houston 2 / T_MFE 7               Houston 4 / T_MFE 14               Houston 2 / T_MFE 14               Houston 10 / T_MFE 5              
model                                     Baseline Patient-aware            Baseline Patient-aware            Baseline Patient-aware             Baseline Patient-aware             Baseline Patient-aware             Baseline Patient-aware
objective, flow                         384,646.90  1,035,503.53          386,896.44  1,068,850.99          390,014.12  1,094,015.80           422,055.32  1,417,544.22           441,561.62  1,468,360.06           377,999.30    950,014.09
objective, reformulation                384,646.90  1,035,503.53          386,896.44  1,068,850.99          390,014.12  1,094,015.80           422,055.32  1,417,544.22           441,561.62  1,468,360.06           377,999.30    950,014.09
total cost, flow                        384,646.

## 6. Equivalence check
- **Objective** values must match within 1e-6 relative — this is the test.
- **E_D** may differ when the objective has several optimal plans (ties); with matching objectives this is
  reported as a tie difference, not an error.
- Extra check: the plan stored by the flow model must be an admissible plan of the reformulation
  (every patient's choice exists in `X`, capacity holds) and get the same cost and E_D from the
  reformulation's coefficients.

In [7]:
mode_code = {"fast": "j1", "slow": "j2"}
eq_rows, all_ok = [], True
for name, d in inst.items():
    plans = enumerate_plans(d)
    for which in MODELS:
        f, r = FLOW[name][which], RES[name, which]
        if r["status"] != "optimal":
            eq_rows.append({"scenario": name, "model": which, "objective": "NOT SOLVED"})
            all_ok = False
            continue
        rel = abs(f["objective"] - r["objective"]) / max(1.0, abs(f["objective"]))
        obj_ok = rel <= 1e-6
        d_ed = r["E_D"] - f["E_D"]
        ed_note = "same E_D" if abs(d_ed) <= 1e-6 else ("tie difference" if obj_ok else "differs (objective mismatch)")
        # flow plan evaluated with the reformulation's data
        fp = f["plan"]
        keys = [(row.p, row.facility, mode_code[row["out mode"]], mode_code[row["return mode"]], int(row.start)) for _, row in fp.iterrows()]
        admissible = all(k in plans for k in keys) and len(keys) == len(d["P"])
        cap_ok = admissible and all(
            sum(1 for k in keys if k[1] == m and t - d["T_MFE"] + 1 <= k[4] <= t) <= d["FCAP"][m]
            for m in d["M"] for t in range(1, d["NT"] + 1))
        same_vals = admissible and abs(sum(plans[k]["cost"] for k in keys) - f["cost"]) <= 1e-4 \
            and abs(sum(plans[k]["riskD"] for k in keys) - f["E_D"]) <= 1e-6 \
            and all(plans[k]["delivery"] == int(row.delivery) and plans[k]["wait"] == int(row.W_p) for k, (_, row) in zip(keys, fp.iterrows()))
        flow_ok = admissible and cap_ok and same_vals
        all_ok &= obj_ok and flow_ok
        eq_rows.append({"scenario": name, "model": which, "objective, flow": f["objective"], "objective, reformulation": r["objective"],
                        "relative difference": rel, "objective": "PASS" if obj_ok else "FAIL",
                        "E_D difference (reformulation − flow)": d_ed, "E_D": ed_note,
                        "flow plan admissible in reformulation, same cost and E_D": "PASS" if flow_ok else "FAIL"})
eq = pd.DataFrame(eq_rows)
eq.to_csv(os.path.join(OUT_DIR, "equivalence_check.csv"), index=False)
print(eq.drop(columns=["objective, flow", "objective, reformulation"]).to_string(
    index=False, formatters={"relative difference": "{:.2e}".format, "E_D difference (reformulation − flow)": "{:+.6f}".format}))
print(f"\nEQUIVALENCE (objective values match within 1e-6 relative in all {len(eq)} cases, "
      f"and every flow plan is admissible with identical cost and E_D): {'PASS' if all_ok else 'FAIL'}")

            scenario         model relative difference objective E_D difference (reformulation − flow)            E_D flow plan admissible in reformulation, same cost and E_D
Houston 10 / T_MFE 7      Baseline            0.00e+00      PASS                             +0.067653 tie difference                                                     PASS
Houston 10 / T_MFE 7 Patient-aware            0.00e+00      PASS                             +0.000000       same E_D                                                     PASS
 Houston 4 / T_MFE 7      Baseline            0.00e+00      PASS                             -0.132988 tie difference                                                     PASS
 Houston 4 / T_MFE 7 Patient-aware            0.00e+00      PASS                             +0.000000       same E_D                                                     PASS
 Houston 2 / T_MFE 7      Baseline            0.00e+00      PASS                             +0.036995 tie difference        

## 7. Verification of every reformulation solution

In [8]:
ver_rows, ver_ok = [], True
for (name, which), lst in VERIF.items():
    for chk, ok in lst:
        ver_ok &= bool(ok)
        ver_rows.append({"scenario": name, "model": which, "check": chk, "result": "PASS" if ok else "FAIL"})
ver = pd.DataFrame(ver_rows)
ver.to_csv(os.path.join(OUT_DIR, "verification.csv"), index=False)
print(ver.pivot_table(index="check", columns=["scenario", "model"], values="result", aggfunc="first", sort=False)
      .reindex(columns=pd.MultiIndex.from_product([list(SCENARIOS), MODELS])).T.to_string())
n_unsolved = sum(1 for r in RES.values() if r["status"] != "optimal")
print(f"\nVERIFICATION ({len(ver)} checks on {len(VERIF)} solutions; {n_unsolved} runs not solved): {'PASS' if ver_ok else 'FAIL'}")
assert ver_ok and all_ok and n_unsolved == 0

check                              one plan per patient capacity respected for every m and t TRT <= ND (40) for every patient timing recomputed from route/modes (arrival, start >= arrival, delivery, TRT, horizon) E_D recomputed from the chosen plans = solver value (1e-6) E_U recomputed from the chosen plans = solver value (1e-6) total cost recomputed from the chosen plans = solver value
Houston 10 / T_MFE 7 Baseline                      PASS                                 PASS                             PASS                                                                                   PASS                                                       PASS                                                       PASS                                                       PASS
                     Patient-aware                 PASS                                 PASS                             PASS                                                                                   PASS        